# EBM Cox Survival Analysis

This notebook covers Cox proportional hazards survival analysis with Explainable Boosting Machines.

Standard regression predicts a scalar target. Survival analysis instead models the *time until an event* (death, failure, churn, ...), where some observations are **censored** — we know only that the event has not happened by a given time. Typical use cases include:

- **Clinical outcomes**: time-to-death, time-to-relapse, time-to-readmission.
- **Reliability**: time-to-failure of hardware components.
- **Customer analytics**: time-to-churn, time-to-purchase.

EBMs support survival modeling via the `"survival_cox"` objective, which fits a Cox proportional hazards model by maximizing the Breslow partial likelihood:

$$\mathcal{L}(\boldsymbol\theta) = \prod_{i \,:\, e_i = 1} \frac{\exp\!\big(f(\mathbf{x}_i)\big)}{\sum_{j \in R(t_i)} \exp\!\big(f(\mathbf{x}_j)\big)}$$

where $R(t_i)$ is the set of subjects still at risk at time $t_i$ (i.e. $t_j \geq t_i$), $e_i \in \{0, 1\}$ indicates an observed event, and $f(\mathbf{x})$ is the additive EBM score — interpretable as the **log hazard ratio** relative to the baseline hazard.

Higher EBM predictions correspond to higher instantaneous hazard, which means *shorter* expected survival.

## Target encoding

A Cox model needs both the observed time and whether the event happened. The EBM API packs both into a single target `y`:

$$y = \text{time} \times (2 \cdot \text{event} - 1)$$

- **Positive `y`**: event was observed at `time = y`.
- **Negative `y`**: observation was censored at `time = -y`.
- **`y = 0`**: rejected (ambiguous).

In [ ]:
# boilerplate
from interpret import show
from interpret.glassbox import ExplainableBoostingRegressor
import numpy as np

def encode_survival(time, event):
    """Pack (time, event) into the signed target used by the Cox objective."""
    time = np.asarray(time, dtype=np.float64)
    event = np.asarray(event, dtype=np.float64)
    return np.where(event > 0.5, time, -time)

## Simulating survival data

We generate a synthetic dataset where the true log hazard depends nonlinearly on two features:

- `feature_0`: linear contribution (coefficient 1.0).
- `feature_1`: quadratic contribution (penalty for extreme values in either direction).

Event times are drawn from an exponential distribution with rate $\exp(\text{log\_hazard})$, and 30% of the samples are randomly right-censored.

In [ ]:
rng = np.random.default_rng(42)
n = 2000

X = rng.normal(size=(n, 2))
true_log_hazard = 1.0 * X[:, 0] + 0.5 * X[:, 1] ** 2

# Exponential event times: larger hazard -> shorter time.
event_times = rng.exponential(scale=np.exp(-true_log_hazard))

# Random right-censoring at 30%.
censored = rng.random(n) < 0.3
event = (~censored).astype(np.float64)

y = encode_survival(event_times, event)

# Train/test split.
split = int(0.8 * n)
X_train, y_train = X[:split], y[:split]
X_test, y_test = X[split:], y[split:]
event_times_test = event_times[split:]
event_test = event[split:]

print(f"Training samples: {len(y_train)}  (events: {int((y_train > 0).sum())},"
      f" censored: {int((y_train < 0).sum())})")
print(f"Test samples:     {len(y_test)}  (events: {int((y_test > 0).sum())},"
      f" censored: {int((y_test < 0).sum())})")

## Fitting the Cox EBM

Use the standard `ExplainableBoostingRegressor` with `objective="survival_cox"`. Everything else (feature binning, term selection, outer bagging) works as usual.

In [ ]:
ebm = ExplainableBoostingRegressor(
    objective="survival_cox",
    interactions=0,
)
ebm.fit(X_train, y_train)

# predict() returns the log hazard ratio: higher -> shorter expected survival.
log_hazard_test = ebm.predict(X_test)
print("First 5 predicted log hazard ratios:", np.round(log_hazard_test[:5], 3))
print("First 5 true      log hazard ratios:",
      np.round(1.0 * X_test[:5, 0] + 0.5 * X_test[:5, 1] ** 2, 3))

## Evaluation: concordance index

The **concordance index** (C-index) is the standard metric for survival ranking quality. It measures the fraction of comparable pairs where the subject with the higher predicted hazard experienced the event first. A C-index of 0.5 is random; 1.0 is perfect ranking.

For the Breslow partial likelihood, a model that perfectly orders samples by their true hazard will maximize the objective — so a higher C-index on held-out data indicates better fit.

In [ ]:
def concordance_index(predictions, times, events):
    """Simple O(n^2) C-index. Higher prediction -> higher hazard -> shorter survival."""
    concordant = 0.0
    comparable = 0
    n = len(predictions)
    for i in range(n):
        if events[i] < 0.5:
            continue
        for j in range(n):
            if i == j or times[j] <= times[i]:
                continue
            comparable += 1
            if predictions[i] > predictions[j]:
                concordant += 1.0
            elif predictions[i] == predictions[j]:
                concordant += 0.5
    return concordant / comparable if comparable else float("nan")


c_index = concordance_index(log_hazard_test, event_times_test, event_test)
print(f"Test C-index: {c_index:.3f}  (0.5 = random, 1.0 = perfect)")

## Recovering the true hazard shape

Because each feature contributes additively to the log hazard ratio, we can read off its effect directly from the shape function. Let's compare the EBM's fitted shape to the true data-generating shape for each feature.

In [ ]:
try:
    import matplotlib.pyplot as plt

    grid = np.linspace(-3, 3, 100)

    # Feature 0: evaluate shape by holding feature_1 at zero.
    X_grid_f0 = np.column_stack([grid, np.zeros_like(grid)])
    pred_f0 = ebm.predict(X_grid_f0) - ebm.intercept_
    true_f0 = 1.0 * grid

    # Feature 1: evaluate shape by holding feature_0 at zero.
    X_grid_f1 = np.column_stack([np.zeros_like(grid), grid])
    pred_f1 = ebm.predict(X_grid_f1) - ebm.intercept_
    true_f1 = 0.5 * grid ** 2

    # Center the true curves so they can be compared to the EBM's zero-centred shapes.
    true_f0 -= true_f0.mean()
    true_f1 -= true_f1.mean()
    pred_f0 -= pred_f0.mean()
    pred_f1 -= pred_f1.mean()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(grid, true_f0, "k--", label="true shape (linear)")
    axes[0].plot(grid, pred_f0, label="EBM shape")
    axes[0].set_title("feature_0 contribution to log hazard")
    axes[0].set_xlabel("feature_0")
    axes[0].set_ylabel("log hazard ratio (centred)")
    axes[0].legend()

    axes[1].plot(grid, true_f1, "k--", label="true shape (quadratic)")
    axes[1].plot(grid, pred_f1, label="EBM shape")
    axes[1].set_title("feature_1 contribution to log hazard")
    axes[1].set_xlabel("feature_1")
    axes[1].set_ylabel("log hazard ratio (centred)")
    axes[1].legend()

    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping plot")

## Risk stratification

A common clinical use of a Cox model is to split patients into risk groups and compare their survival experience. We can stratify the test set by the predicted log hazard ratio and plot a Kaplan–Meier-style empirical survival curve for each group.

In [ ]:
def kaplan_meier(times, events):
    """Return (sorted_times, survival_probability) arrays for a KM estimate."""
    order = np.argsort(times)
    t = times[order]
    e = events[order]
    n_at_risk = len(t)
    surv = 1.0
    xs, ys = [0.0], [1.0]
    for i in range(len(t)):
        if e[i] > 0.5:
            surv *= (n_at_risk - 1) / n_at_risk
        xs.append(t[i])
        ys.append(surv)
        n_at_risk -= 1
    return np.asarray(xs), np.asarray(ys)


tertiles = np.quantile(log_hazard_test, [1 / 3, 2 / 3])
group = np.digitize(log_hazard_test, tertiles)  # 0 = low, 1 = mid, 2 = high hazard

try:
    fig, ax = plt.subplots(figsize=(9, 5))
    labels = ["low risk (bottom tertile)", "mid risk", "high risk (top tertile)"]
    for g in range(3):
        mask = group == g
        x, s = kaplan_meier(event_times_test[mask], event_test[mask])
        ax.step(x, s, where="post", label=labels[g])
    ax.set_xlabel("time")
    ax.set_ylabel("survival probability")
    ax.set_title("Empirical survival curves by EBM-predicted risk tertile")
    ax.legend()
    ax.set_ylim(0, 1.02)
    plt.tight_layout()
    plt.show()
except NameError:
    for g in range(3):
        mask = group == g
        rate = event_test[mask].mean()
        print(f"group {g}: n={mask.sum()}, event rate={rate:.2%}")

The high-risk group should experience events faster (steeper drop), and the low-risk group should retain more survivors. A clean separation between the curves indicates the model has learned a meaningful risk stratification.

## Interpretability

All standard EBM explanations work unchanged — the additive score is now the log hazard ratio. Features with strongly non-zero shape functions move the hazard up or down; features with flat shapes have no effect.

In [ ]:
show(ebm.explain_global())

In [ ]:
# Local explanation for a handful of test samples
show(ebm.explain_local(X_test[:5], y_test[:5]), 0)

## Summary

- Use `objective="survival_cox"` with `ExplainableBoostingRegressor` for survival analysis.
- Encode the target as `y = time * (2 * event - 1)` so that positive values are events and negative values are censored times.
- `predict()` returns the additive log hazard ratio: higher score ↔ higher instantaneous hazard ↔ shorter expected survival.
- The partial likelihood depends only on the *ordering* of event times, so rescaling all times by a constant does not change predictions.
- Every EBM interpretability tool — global shape functions, local score breakdowns, feature importances, merging, pickling — works with Cox exactly as it does for standard regression.